# 05. 시연 사과 묶음 생성과 당도 연결

학습에 선택된 프레임을 보수적으로 제외 → 묶음 생성 → SHA·출처 검증 → 고정 가상 당도 연결 순서입니다. 같은 실제 사과의 다른 사진이므로 독립 평가 데이터가 아닙니다.

실행 결과는 저장하지 않았습니다. 아래 실행 스위치는 기본 `False`이며, 전체 실행만으로 학습·파일 생성·서버 검사가 시작되지 않습니다.

In [ ]:
from pathlib import Path
import os
import sys
import json

# 저장소 루트 또는 notebooks 폴더에서 실행할 수 있습니다.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src/data').is_dir() and (p / 'configs').is_dir()), None)
if ROOT is None:
    raise RuntimeError('CQC 저장소 안에서 노트북을 열어 주세요.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('작업 폴더:', ROOT)


## 시연 묶음 생성

서버 학습 설정과 매니페스트 SHA가 일치해야 합니다. execute=False로 먼저 예상 수량을 확인하세요.

In [ ]:
"""기존 학습에서 선택될 수 있었던 사진을 보수적으로 제외한 뒤 시연 묶음을 만든다."""
from __future__ import annotations

import csv
import hashlib
import json
import random
import shutil
from pathlib import Path
from zipfile import ZipFile

from data.sampling.angle_balanced import select_angle_balanced_random_views
from src.data.multiview import load_groups, select_views


In [ ]:
def write_json(path, value):
    """한글을 유지하는 JSON으로 결과를 저장한다."""
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


In [ ]:
def selection_exclusions(groups, configs):
    """기록된 모든 사진 선택 설정을 전체 사과와 예정된 모든 학습 회차에 적용해 제외 목록을 만든다."""
    signatures = set()
    for config in configs:
        sampling = config.get("sampling") or "fixed"
        if sampling not in {"fixed", "angle_balanced_random"}:
            raise ValueError(f"Unrecognized historical sampling: {sampling}")
        signatures.add((sampling, int(config["views"]), int(config["seed"]),
                        int(config["epochs"]) if sampling != "fixed" else 1))
    excluded = set()
    for group in groups:
        for sampling, views, seed, epochs in sorted(signatures):
            for epoch in range(1, epochs + 1):
                selected = (select_views(group.frames, views) if sampling == "fixed" else
                            select_angle_balanced_random_views(group.frames, views,
                                seed=seed, epoch=epoch, group_no=group.group_no))
                excluded.update(frame.sample_id for frame in selected.real_frames)
    return excluded, sorted(signatures)


In [ ]:
def make_bundles(groups, excluded):
    """남은 사진을 같은 사과끼리 묶고 각도 균형을 맞춰 최대 12장씩 나눈다."""
    bundles = []
    for group in groups:
        remaining = [frame for frame in group.frames if frame.sample_id not in excluded]
        index = 0
        while remaining:
            selected = select_angle_balanced_random_views(remaining, min(12, len(remaining)),
                seed=42, epoch=index, group_no=group.group_no).real_frames
            bundles.append((group, index, selected))
            ids = {frame.sample_id for frame in selected}
            remaining = [frame for frame in remaining if frame.sample_id not in ids]
            index += 1
    return bundles


In [ ]:
def build_demo_bundles(*, evidence, output=Path('data/processed/realtime-apple-arrival-demo'), manifest=Path('data/processed/manifest.csv'), splits=Path('configs/splits/seed-42.csv'), raw_root=Path('data/raw'), execute=False):
    """학습 근거와 출처를 확인하고 시연 사진·요청·출처 파일을 생성한다."""
    evidence = json.loads(evidence.read_text(encoding="utf-8"))
    digest = hashlib.sha256(manifest.read_bytes()).hexdigest()
    if digest.lower() != evidence["manifest_sha256"].lower():
        raise ValueError("Local and training-server manifests differ")
    groups = load_groups(manifest, splits)
    with manifest.open(encoding="utf-8-sig", newline="") as stream:
        rows = {row["sample_id"]: row for row in csv.DictReader(stream)}
    excluded, signatures = selection_exclusions(groups, evidence["configs"])
    # 같은 사진 바이트는 CRC32와 길이도 같으므로 같은 조합을 보수적으로 제외한다.
    # CRC32 충돌도 제외 대상에 포함하며, CRC32를 고유 이미지 식별자로 보지는 않는다.
    blocked_content = {(rows[key]["image_crc32"], rows[key]["image_size_bytes"]) for key in excluded}
    excluded.update(key for key, row in rows.items()
                    if (row["image_crc32"], row["image_size_bytes"]) in blocked_content)
    bundles = make_bundles(groups, excluded)
    exported_ids = [frame.sample_id for _, _, frames in bundles for frame in frames]
    if len(exported_ids) != len(set(exported_ids)) or set(exported_ids) & excluded:
        raise ValueError("Repeated or excluded source frame in demo export")
    summary = {
        "purpose": "demo_stream_only_not_training_or_independent_evaluation",
        "same_physical_apples_as_original_dataset": True,
        "exclusion_policy": "all_recorded_selectors_on_all_groups_and_all_planned_epochs",
        "exact_per_batch_consumption_log_available": False,
        "manifest_sha256": digest,
        "training_configs_checked": len(evidence["configs"]),
        "selection_signatures": signatures,
        "total_original_images": len(rows), "excluded_images": len(excluded),
        "exported_images": len(exported_ids),
        "source_apples": len({group.group_no for group, _, _ in bundles}),
        "full_12_frame_bundles": sum(len(frames) == 12 for _, _, frames in bundles),
        "partial_bundles": sum(len(frames) < 12 for _, _, frames in bundles),
        "excluded_overlap": 0,
        "estimated_image_bytes": sum(int(rows[key]["image_size_bytes"]) for key in exported_ids),
    }
    print(json.dumps(summary), flush=True)
    if not execute:
        return
    if output.exists():
        raise FileExistsError(f"Existing destination is preserved: {output}")
    if shutil.disk_usage(output.parent).free < summary["estimated_image_bytes"] + 1024**3:
        raise OSError("Insufficient space for demo export")
    output.mkdir(parents=True)
    write_json(output / "source-training-configs.json", evidence)
    write_json(output / "excluded-sample-ids.json", sorted(excluded))
    archives = {}
    index = []
    try:
        for number, (group, bundle_index, frames) in enumerate(bundles, 1):
            demo_id = f"demo-{group.group_no}-{bundle_index:03d}"
            relative = Path("groups" if len(frames) == 12 else "partial_groups") / demo_id
            destination = output / relative
            destination.mkdir(parents=True)
            metadata, images, provenance = [], [], []
            for view_index, frame in enumerate(frames):
                archive_path = (raw_root / frame.image_archive).resolve()
                if raw_root.resolve() not in archive_path.parents:
                    raise ValueError("Archive outside raw root")
                if archive_path not in archives:
                    archives[archive_path] = ZipFile(archive_path)
                value = archives[archive_path].read(frame.image_member)
                name = f"frame_{view_index:02d}.png"
                with (destination / name).open("xb") as stream:
                    stream.write(value)
                images.append(name)
                metadata.append(dict(view_index=view_index, angle_direction=frame.angle_direction,
                    verticality_angle=frame.verticality_angle, horizontality_angle=frame.horizontality_angle))
                provenance.append(dict(file=name, sample_id=frame.sample_id,
                    source_archive=frame.image_archive, source_member=frame.image_member,
                    sha256=hashlib.sha256(value).hexdigest(), bytes=len(value)))
            write_json(destination / "request.json", dict(inspection_id=demo_id, images=images, metadata=metadata))
            write_json(destination / "provenance.json", dict(source_group_id=group.group_no,
                original_split=group.split, same_physical_apple=True, frames=provenance))
            write_json(destination / "expected.json", dict(cultivar=group.cultivar,
                quality_grade=group.quality_grade, send_to_model=False))
            index.append(dict(inspection_id=demo_id, path=relative.as_posix(),
                source_group_id=group.group_no, frame_count=len(frames), default_playback=len(frames)==12))
            if number % 50 == 0:
                print(f"exported_bundles={number}/{len(bundles)}", flush=True)
    finally:
        for archive in archives.values():
            archive.close()
    random.Random(42).shuffle(index)
    write_json(output / "index.json", index)
    write_json(output / "summary.json", summary)
    print(f"completed output={output}", flush=True)


In [ ]:
RUN_BUILD_DEMO_BUNDLES = False
if RUN_BUILD_DEMO_BUNDLES:
    build_demo_bundles(evidence=Path("data/processed/realtime-apple-arrival-demo/source-training-configs.json"), output=Path("data/processed/demo-new"), execute=False)


## 묶음 무결성 검증

생성한 폴더를 지정하세요. 원본 sample_id·각도·SHA·중복·제외 여부를 검사합니다.

In [ ]:
"""시연 사진의 출처·해시·요청 각도와 학습 사진 제외 여부를 검사한다."""
import csv
import hashlib
import json
from collections import Counter
from pathlib import Path


In [ ]:
def verify_demo_bundles(*, root=Path('data/processed/realtime-apple-arrival-demo'), manifest=Path('data/processed/manifest.csv')):
    """시연 묶음의 사진 해시·원본 그룹·각도·중복과 학습 제외 여부를 검사한다."""
    read = lambda path: json.loads(path.read_text(encoding="utf-8"))
    summary = read(root / "summary.json")
    index = read(root / "index.json")
    excluded = set(read(root / "excluded-sample-ids.json"))
    with manifest.open(encoding="utf-8-sig", newline="") as stream:
        source = {row["sample_id"]: row for row in csv.DictReader(stream)}
    seen = set()
    hashes = Counter()
    total_bytes = 0
    for number, bundle in enumerate(index, 1):
        path = (root / bundle["path"]).resolve()
        assert root.resolve() in path.parents
        request = read(path / "request.json")
        provenance = read(path / "provenance.json")
        assert set(request) == {"inspection_id", "images", "metadata"}
        assert request["inspection_id"] == bundle["inspection_id"]
        assert len(request["images"]) == len(request["metadata"]) == len(provenance["frames"]) == bundle["frame_count"]
        for i, frame in enumerate(provenance["frames"]):
            key = frame["sample_id"]
            assert key not in excluded and key not in seen
            seen.add(key)
            row = source[key]
            assert row["source_group_id"] == provenance["source_group_id"] == bundle["source_group_id"]
            assert request["images"][i] == frame["file"]
            payload = (path / frame["file"]).read_bytes()
            digest = hashlib.sha256(payload).hexdigest()
            assert digest == frame["sha256"] and len(payload) == int(row["image_size_bytes"]) == frame["bytes"]
            assert payload.startswith(b"\x89PNG\r\n\x1a\n")
            hashes[digest] += 1
            total_bytes += len(payload)
            assert request["metadata"][i] == dict(view_index=i, angle_direction=row["angle_direction"],
                verticality_angle=int(row["verticality_angle"]), horizontality_angle=int(row["horizontality_angle"]))
        if number % 200 == 0:
            print(f"verified={number}/{len(index)}", flush=True)
    assert len(seen) == summary["exported_images"]
    assert total_bytes == summary["estimated_image_bytes"]
    report = dict(verified_images=len(seen), verified_bundles=len(index), excluded_overlap=0,
        repeated_source_sample_ids=0, repeated_content_hashes=sum(count-1 for count in hashes.values()),
        total_image_bytes=total_bytes, metadata_matches_source=True)
    (root / "verification.json").write_text(json.dumps(report, indent=2)+"\n", encoding="utf-8")
    print(json.dumps(report))


In [ ]:
RUN_VERIFY_DEMO_BUNDLES = False
if RUN_VERIFY_DEMO_BUNDLES:
    verify_demo_bundles(root=Path("data/processed/demo-new"))


## 묶음별 고정 가상 당도 연결

group_no의 가상 당도를 demo_bundle_id와 연결합니다. 기존 파일은 덮어쓰지 않습니다.

In [ ]:
"""시연 묶음 목록에 사과별로 고정된 가상 당도를 연결한다."""

from __future__ import annotations

import csv
import json
from pathlib import Path


In [ ]:
def attach_demo_brix(*, demo_root=Path('data/processed/realtime-apple-arrival-demo'), virtual_brix=Path('data/processed/virtual-brix.csv')):
    """시연 묶음 식별자에 가상 당도를 연결하고 14도 기준 구간을 기록한다."""
    output = demo_root / "demo-virtual-brix.csv"
    if output.exists():
        raise ValueError(f"Refusing to overwrite {output}")

    index = json.loads((demo_root / "index.json").read_text(encoding="utf-8"))
    with virtual_brix.open(newline="", encoding="utf-8-sig") as stream:
        by_group = {row["group_no"]: row for row in csv.DictReader(stream)}
    if len(by_group) == 0 or len({item["inspection_id"] for item in index}) != len(
        index
    ):
        raise ValueError("Missing Brix source or duplicate demo bundle ID")

    rows = []
    for item in index:
        source = by_group[item["source_group_id"]]
        if source["brix_is_measured"].lower() != "false":
            raise ValueError("Demo Brix must be explicitly marked unmeasured")
        value = float(source["virtual_brix"])
        if not 9 <= value <= 18:
            raise ValueError("Virtual Brix outside 9..18")
        rows.append(
            {
                "demo_bundle_id": item["inspection_id"],
                "virtual_brix": f"{value:.1f}",
                "sweetness_band": "sweet" if value >= 14.0 else "less_sweet",
                "brix_source": source["brix_source"],
                "brix_is_measured": "false",
                "brix_generator_version": source["brix_generator_version"],
            }
        )
    with output.open("x", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    print(
        f"demo_brix={output} bundles={len(rows)} less_sweet={sum(row['sweetness_band'] == 'less_sweet' for row in rows)}"
    )


In [ ]:
RUN_ATTACH_DEMO_BRIX = False
if RUN_ATTACH_DEMO_BRIX:
    attach_demo_brix(demo_root=Path("data/processed/demo-new"))
